In [10]:
!pip install -q -U langgraph langchain-google-genai

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR API KEY HERE"

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)

In [14]:
#State definition

class State(TypedDict):
  name: str
  message: str

In [15]:
#Node Definition

def greeting_node(state: State):
    return {
        "message": "Hello " + state["name"] + "!"
    }


def namaste_node(state: State):
    return {
        "message": state["message"] + "\nnamaste " + state["name"] + "!"
    }


def hola_node(state: State):
    return {
        "message": state["message"] + "\nHola " + state["name"] + "!"
    }

In [16]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

# Add nodes
graph.add_node("greeting", greeting_node)
graph.add_node("namaste", namaste_node)
graph.add_node("hola", hola_node)

# Add edges
graph.add_edge(START, "greeting")
graph.add_edge("greeting", "namaste")
graph.add_edge("namaste", "hola")
graph.add_edge("hola", END)

In [17]:
#Compile

app = graph.compile()

In [18]:
name = input("Enter your name: ")

result = app.invoke({
    "name": name,
    "message": ""
})

print(result["message"])

Enter your name: Radha
Hello Radha!
namaste Radha!
Hola Radha!


In [19]:
class State(TypedDict):
    question: str
    answer: str

In [20]:
# Create LLM Node

def llm_node(state: State):
    response = llm.invoke(
        state["question"]
    )

    return {
        "answer": response.content
    }

In [21]:
# Build graph

graph = StateGraph(State)

# Add nodes
graph.add_node("llm", llm_node)

# Add edges
graph.add_edge(START, "llm")
graph.add_edge("llm", END)

In [22]:
#compile

app = graph.compile()

In [23]:
question = input("Enter your question: ")

result = app.invoke({
    "question": question
})

print(result["answer"])

Enter your question: Tell about Politics in India in 1 line


[{'type': 'text', 'text': 'Indian politics is a vibrant, complex, and high-stakes arena defined by deep diversity, intense electoral competition, and the democratic voice of over 1.4 billion people.', 'extras': {'signature': 'Et4QCtsQAWkUfRMpo8v60mpneTohCh5a880NbJwaCx35lWah1rMRZ/yUCzE87FxcXBhtJz9C2uejolJ8H4zf5RtCItOkGjhfMqz4gXcw/9HtLUB3hZBjOXQbH/FO40tyx1FCn8amQKTqgfpfw1BYmenAh/EsOUBvxUFypbWw+jJgvczR3GkJtJJuZJy/jPV2wLGLjaXj18sMR3FwBjagGoZ343Za9QicHjkQHALScAFxupLsqxE22fQiFbcJKdoraj7DhRY2hSOmJ5A/FaW1+Nkfxo5RKc6KoRuS3q3EI22aJ35j5nf1JLKXJAcvEUHskB8voH6pRVnj1jnIORQJWl3TIgf5EtFc1RjXzKOlq9GiA4VPjVtPqWZUj9AJf8itPt+BK2W10a6ldeKbpVw3c6f3Y6JNj465wGk0tIGD4kDxzF8sirWH1RVRJFPveWNefpkBEAP1vtb3U16w9z7NFrdCf5VjerTrLykqtbU14qsownyXewwR8GuyXO7STl1Y3pjc36y12DUFD+1ww8msihVuXTWudaV27A/96518Vp2MbsuggLmZgzG3JpBcil+xZUuLuvFVNWLWkquz4tMpLx+LgL/BeK0ir9EdgP5fxZzEY/sJL/NW/PEsee9rEdMGeLIJi8dAuRki19B1u9ysk9uA4SltxI+SP3CkzxM0d303BjLBLC2s1jI4Njn5gTJ3IWqV/cqCemUpcnpb3cT2/5tis3Gxe8VBsCAjfIbldMdSNmbC9YKoFiOZvGEXD73HaOrHRI